In [ ]:
# FHIR Bronze Layer
# Converts immutable Raw JSON Bundles into append-only Delta tables.
# JSON-to-table conversion is performed with Spark, not Dataflow Gen2.


In [ ]:
extraction_date = "2026-10-14"
resources = "Patient,Encounter,Observation,Condition"

In [3]:
RESOURCES = [r.strip() for r in resources.split(",") if r.strip()]

StatementMeta(, 8bdda1df-f325-4ee4-9897-59170dad43c6, 5, Finished, Available, Finished, False)

In [1]:
from pyspark.sql import functions as F

RAW_BASE_PATH = "Files/raw"
RESOURCES = ["Patient", "Encounter", "Observation", "Condition"]

for resource in RESOURCES:
    source_path = f"{RAW_BASE_PATH}/{resource}/{extraction_date}/"

    raw_df = (
        spark.read
        .option("multiLine", "true")
        .json(source_path)
    )

    bronze_df = (
        raw_df
        .select(
            F.input_file_name().alias("source_file"),
            F.explode_outer("entry").alias("entry")
        )
        .filter(F.col("entry.resource").isNotNull())
        .select("source_file", F.col("entry.resource.*"))
        .withColumn("resource_type", F.lit(resource))
        .withColumn("extraction_timestamp",  F.to_timestamp(F.lit(extraction_timestamp)))
        .withColumn("bronze_loaded_timestamp", F.current_timestamp())
        .join(
            spark.table("raw_ingestion_log")
                 .select("source_file", "api_url_or_params")
                 .dropDuplicates(["source_file"]),
            on="source_file",
            how="left"
        )
    )

    table_name = f"bronze_{resource.lower()}"

    (bronze_df.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(table_name))

    print(f"Loaded {table_name}: {bronze_df.count()} rows")

print("Raw -> Bronze conversion complete.")


StatementMeta(, 8bdda1df-f325-4ee4-9897-59170dad43c6, 3, Finished, Available, Finished, False)

Loaded bronze_patient: 5820 rows
Loaded bronze_encounter: 2256 rows
Loaded bronze_observation: 36629 rows
Loaded bronze_condition: 2772 rows
Raw -> Bronze conversion complete.
